# 02b — Method 2: Chain-of-Thought LLM

Feature Selection + Range Prediction

Single LLM call that:
1. Interprets the text prompt to identify desired musical qualities
2. Selects relevant features from our catalog
3. Predicts target ranges (min, max) for each selected feature
4. Self-validates the reasoning — refines if the combination doesn't make sense
5. Returns structured JSON

Songs are then scored by how well their feature values fall within the predicted ranges.

**Prerequisites:** Run `02a_data_feature_prep.ipynb` first.

In [14]:
# --- Shared Setup ---
# This notebook depends on the data and features prepared in 02a.
# Run 02a_data_feature_prep.ipynb first, or load saved artifacts.

import sys
sys.path.insert(0, '..')

import os
import json
import time
import warnings
from pathlib import Path
from typing import TypedDict, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.spatial.distance import cosine as cosine_dist
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore', category=FutureWarning)

from atdj.config import PROCESSED_DIR, DATA_DIR

# --- Load saved artifacts from notebook 02a ---
FEATURES_DIR = Path(PROCESSED_DIR) / 'features_exp'
df_merged = pd.read_pickle(FEATURES_DIR / 'df_merged.pkl')
print(f'Loaded df_merged: {df_merged.shape}')

import pickle
with open(FEATURES_DIR / 'feature_catalog.pkl', 'rb') as f:
    FEATURE_CATALOG = pickle.load(f)

ALL_FEATURE_NAMES = []
FEATURE_DESC_MAP = {}
for group, features in FEATURE_CATALOG.items():
    for fname, fdesc in features:
        ALL_FEATURE_NAMES.append(fname)
        FEATURE_DESC_MAP[fname] = fdesc
AVAILABLE_FEATURES = [f for f in ALL_FEATURE_NAMES if f in df_merged.columns]

with open(FEATURES_DIR / 'feature_prompt.txt', encoding='utf-8') as f:
    FEATURE_PROMPT = f.read()

print(f'Feature catalog: {len(AVAILABLE_FEATURES)} available features across {len(FEATURE_CATALOG)} groups')

Loaded df_merged: (184, 213)
Feature catalog: 99 available features across 15 groups


In [ ]:
# --- LLM Setup ---
try:
    from google import genai
    from google.genai import types as genai_types
    GEMINI_AVAILABLE = True
except ImportError:
    GEMINI_AVAILABLE = False
    print('google-genai SDK not available')

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
GEMINI_MODEL = os.getenv('GEMINI_MODEL', 'gemini-2.0-flash')
LLM_READY = GEMINI_AVAILABLE and GEMINI_API_KEY is not None
if LLM_READY:
    print(f'LLM ready: model={GEMINI_MODEL}')
else:
    print('LLM not ready — set GEMINI_API_KEY')


def llm_chat(client, system: str, user_message: str, max_tokens: int = 2000) -> str:
    import random
    max_retries = 8
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=user_message,
                config=genai_types.GenerateContentConfig(
                    system_instruction=system if system else None,
                    max_output_tokens=max_tokens,
                ),
            )
            return response.text.strip()
        except Exception as e:
            if ('429' in str(e) or 'RESOURCE_EXHAUSTED' in str(e)) and attempt < max_retries - 1:
                wait = min(30 * (2 ** attempt), 300) + random.uniform(0, 10)
                print(f'  Rate limited, waiting {wait:.0f}s (attempt {attempt+1}/{max_retries})...')
                time.sleep(wait)
            else:
                raise

In [ ]:
# Standardized output format — all methods return this structure
class SongMatch(TypedDict):
    filename: str
    score: float              # 0-1, higher = better match
    matched_features: dict    # feature_name: actual_value
    explanation: str          # why this song matches

class MatchResult(TypedDict):
    prompt: str
    method: str               # 'clap' | 'cot_llm_a' | 'cot_llm_b' | 'small_model'
    top_k_songs: list         # list of SongMatch
    bottom_k_songs: list      # worst-matching songs for human validation
    feature_ranges_used: dict # feature_name: {min, max, direction} (methods 2 & 3)
    metadata: dict            # method-specific (model name, latency, etc.)


# Test prompts used across all methods
TEST_PROMPTS = [
    'energetic tango with strong rhythm for experienced dancers',
    'melancholic and slow, perfect for a late-night vals',
    'bright and cheerful milonga with clear melody',
    'dramatic tango with heavy bandoneon and dark mood',
    'smooth and relaxed, good for warming up the floor',
    'classic golden-age tango from the 40s, warm and nostalgic',
    'a lively vals from the 50s with a strong orchestra',
    'need a cortina — something upbeat and non-tango to reset the floor',
]

TOP_K = 10
BOTTOM_K = 5  # worst matches for human validation  # number of songs to return per query

# Collect all results here for cross-method comparison
ALL_RESULTS: dict[str, list[MatchResult]] = {}  # method_name -> list of results

print(f'Test prompts: {len(TEST_PROMPTS)}')
print(f'Top-K: {TOP_K}')
for i, p in enumerate(TEST_PROMPTS, 1):
    print(f'  {i}. "{p}"')

Test prompts: 8
Top-K: 10
  1. "energetic tango with strong rhythm for experienced dancers"
  2. "melancholic and slow, perfect for a late-night vals"
  3. "bright and cheerful milonga with clear melody"
  4. "dramatic tango with heavy bandoneon and dark mood"
  5. "smooth and relaxed, good for warming up the floor"
  6. "classic golden-age tango from the 40s, warm and nostalgic"
  7. "a lively vals from the 50s with a strong orchestra"
  8. "need a cortina — something upbeat and non-tango to reset the floor"


In [ ]:
COT_SYSTEM_PROMPT = """You are an expert Argentine Tango DJ and audio engineer.
You help match text descriptions of desired music to songs using audio features.

When given a text prompt and a list of available audio features with their observed ranges,
you must select the most relevant features and predict target value ranges that would
match the described music."""

COT_USER_TEMPLATE = """Given a user's text prompt describing desired tango music, select the most
relevant audio features and specify target ranges for song matching.

## Available Features (with observed ranges in our 100-song catalog):
{feature_descriptions}

## User Prompt: "{prompt}"

## Instructions:
Think step by step:
1. What musical qualities does this prompt describe?
2. Which features from the list above best capture those qualities? Select 5-10 features.
3. For each selected feature, what value range would match the prompt? Use the observed ranges as guidance.
4. Validate your reasoning: does this combination of features and ranges make musical sense together?
   If not, refine your selections.

## Response Format:
Respond ONLY with valid JSON (no markdown, no code blocks):
{{
  "reasoning": "Brief explanation of your musical interpretation and feature choices",
  "selected_features": {{
    "feature_name": {{
      "min": <float>,
      "max": <float>,
      "direction": "higher_better" | "lower_better" | "range",
      "weight": <float 0-1, importance of this feature>
    }}
  }},
  "validation": "Brief validation that this combination makes sense"
}}"""


def cot_select_and_range(prompt: str, feature_desc: str, client) -> dict:
    """Ask LLM to select features and predict ranges via CoT.
    Returns parsed JSON with selected_features and reasoning."""
    user_message = COT_USER_TEMPLATE.format(
        feature_descriptions=feature_desc,
        prompt=prompt
    )
    response_text = llm_chat(client, COT_SYSTEM_PROMPT, user_message, max_tokens=2000)
    # Parse JSON -- handle potential markdown code blocks
    if response_text.startswith('```'):
        response_text = response_text.split('```')[1]
        if response_text.startswith('json'):
            response_text = response_text[4:]
    try:
        return json.loads(response_text)
    except json.JSONDecodeError as e:
        print(f'  WARNING: Failed to parse LLM response as JSON: {e}')
        print(f'  Response: {response_text[:500]}')
        return {'selected_features': {}, 'reasoning': 'Parse error', 'validation': 'N/A'}


def score_songs_by_ranges(df: pd.DataFrame, feature_ranges: dict) -> pd.Series:
    """Score each song by how well it fits the predicted feature ranges.
    Score per feature: 1.0 if value in [min,max], linear decay outside.
    Final score: weighted average across selected features."""
    scores = pd.Series(0.0, index=df.index)
    total_weight = 0.0

    for fname, spec in feature_ranges.items():
        if fname not in df.columns:
            continue
        col = df[fname]
        if col.dtype not in ['float64', 'float32', 'int64']:
            continue

        fmin = spec.get('min', col.min())
        fmax = spec.get('max', col.max())
        weight = spec.get('weight', 1.0)
        range_width = fmax - fmin if fmax > fmin else 1.0

        # Score: 1.0 inside range, linear decay outside
        feature_score = pd.Series(0.0, index=df.index)
        inside = (col >= fmin) & (col <= fmax)
        feature_score[inside] = 1.0

        below = col < fmin
        if below.any():
            dist = (fmin - col[below]) / range_width
            feature_score[below] = np.maximum(0, 1.0 - dist)

        above = col > fmax
        if above.any():
            dist = (col[above] - fmax) / range_width
            feature_score[above] = np.maximum(0, 1.0 - dist)

        scores += feature_score * weight
        total_weight += weight

    if total_weight > 0:
        scores /= total_weight

    return scores


def rank_songs_cot_a(prompt: str, df: pd.DataFrame, feature_desc: str,
                     client, top_k: int = 10, bottom_k: int = 5) -> MatchResult:
    """Full pipeline for Method 2A: CoT feature selection + range scoring."""
    t0 = time.time()

    # Step 1: LLM selects features and ranges
    llm_output = cot_select_and_range(prompt, feature_desc, client)
    feature_ranges = llm_output.get('selected_features', {})

    # Step 2: Score songs
    scores = score_songs_by_ranges(df, feature_ranges)
    elapsed = time.time() - t0

    # Build result — top and bottom matches
    def _build_songs(indices):
        songs = []
        for idx in indices:
            row = df.loc[idx]
            matched = {}
            for fname in feature_ranges:
                if fname in row.index and pd.notna(row[fname]):
                    try:
                        matched[fname] = round(float(row[fname]), 4)
                    except (ValueError, TypeError):
                        pass
            songs.append({
                'filename': row['filename'],
                'score': round(float(scores[idx]), 4),
                'matched_features': matched,
                'explanation': llm_output.get('reasoning', ''),
            })
        return songs

    top_songs = _build_songs(scores.nlargest(top_k).index)
    bottom_songs = _build_songs(scores.nsmallest(bottom_k).index)

    return {
        'prompt': prompt,
        'method': 'cot_llm_a',
        'top_k_songs': top_songs,
        'bottom_k_songs': bottom_songs,
        'feature_ranges_used': feature_ranges,
        'metadata': {
            'model': GEMINI_MODEL,
            'reasoning': llm_output.get('reasoning', ''),
            'validation': llm_output.get('validation', ''),
            'latency_seconds': round(elapsed, 3),
            'num_features_selected': len(feature_ranges),
        },
    }


print('Method 2A functions defined.')
if not LLM_READY:
    print('NOTE: Set GEMINI_API_KEY in .env to run this method.')

Method 2A functions defined.


In [24]:
# Run Method 2A across all test prompts
if LLM_READY:
    client = genai.Client(api_key=GEMINI_API_KEY)
    cot_a_results = []
    for i, prompt in enumerate(TEST_PROMPTS):
        print(f'\nPrompt: "{prompt}"')
        result = rank_songs_cot_a(prompt, df_merged, FEATURE_PROMPT, client, top_k=TOP_K, bottom_k=BOTTOM_K)
        cot_a_results.append(result)
        ranges = result['feature_ranges_used']
        print(f'  Features selected ({len(ranges)}):')
        for fname, spec in ranges.items():
            lo, hi = spec.get('min', '?'), spec.get('max', '?')
            w = spec.get('weight', '?')
            print(f'    {fname}: [{lo}, {hi}]  weight={w}')
        print(f'  Top 3: {[s["filename"] for s in result["top_k_songs"][:3]]}')
        print(f'  Bottom 3: {[s["filename"] for s in result["bottom_k_songs"][:3]]}')
        print(f'  Latency: {result["metadata"]["latency_seconds"]}s')
        if i < len(TEST_PROMPTS) - 1:
            time.sleep(5)  # pace requests to avoid rate limits

    ALL_RESULTS['cot_llm_a'] = cot_a_results
    print(f'--- Method 2A complete: {len(cot_a_results)} prompts processed ---')
else:
    print('Method 2A skipped — set GEMINI_API_KEY in .env')
    ALL_RESULTS['cot_llm_a'] = []



Prompt: "energetic tango with strong rhythm for experienced dancers"
  Features selected (7):
    bpm: [125.0, 160.0]  weight=0.9
    onset_rate: [3.4, 4.2]  weight=0.8
    average_loudness: [0.65, 0.95]  weight=0.7
    dynamic_complexity: [5.0, 8.0]  weight=0.6
    danceability: [1.0, 1.3]  weight=0.8
    mood_happy: [0.3, 0.6]  weight=0.4
    tempogram_global_mean: [0.21, 0.26]  weight=0.5
  Top 3: ['21 Cancion De Rango.mp3', '06 Malena.mp3', '03 No Le Digas Que La Quiero.mp3']
  Bottom 3: ['08 Royal Pigall.mp3', '25 Cortina.mp3', '04 La Viruta.mp3']
  Latency: 4.207s

Prompt: "melancholic and slow, perfect for a late-night vals"
  Rate limited, waiting 30s (attempt 1/6)...
  Features selected (6):
    style: [vals, vals]  weight=1.0
    bpm: [90.0, 115.0]  weight=0.9
    mood_happy: [0.0, 0.25]  weight=0.8
    mirex_mood_literate_poignant_wistful_bittersweet_autumnal_brooding: [0.4, 0.75]  weight=0.7
    average_loudness: [0.3, 0.7]  weight=0.5
    chords_changes_rate: [0.03, 0.07]

In [ ]:
# --- Save Method 2A results for downstream notebooks ---
results_dir = Path(PROCESSED_DIR) / 'features_exp'
results_dir.mkdir(parents=True, exist_ok=True)

def _jsonify(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, dict):
        return {k: _jsonify(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_jsonify(v) for v in obj]
    return obj

method_name = 'cot_llm_a'
if ALL_RESULTS.get(method_name):
    path = results_dir / f'{method_name}_results.json'
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(_jsonify(ALL_RESULTS[method_name]), f, indent=2)
    print(f"Saved {len(ALL_RESULTS[method_name])} {method_name} results to {path.name}")

Saved 8 cot_llm_a results to cot_llm_a_results.json

Generating additional prompts for small model training...
Labelling 51 extra prompts via Method 2A...
  Response: 
{
  "reasoning": "The prompt describes romantic tango music with lyrics about lost love. This suggests a slower tempo, a melancholic mood, and possibly more emphasis on vocals and traditional instruments. I'll choose features that reflect these aspects.",
  "selected_features": {
    "bpm": {
      "min": 90.0,
      "max": 125.0,
      "direction": "lower_better",
      "weight": 0.8
    },
    "mood_happy": {
      "min": 0.002,
      "max": 0.35,
      "direction": "lower_better",
      "wei
  Rate limited, waiting 30s (attempt 1/6)...
  Rate limited, waiting 30s (attempt 1/6)...
  Rate limited, waiting 60s (attempt 2/6)...
  Rate limited, waiting 90s (attempt 3/6)...
  Rate limited, waiting 30s (attempt 1/6)...
  Rate limited, waiting 30s (attempt 1/6)...
  Rate limited, waiting 60s (attempt 2/6)...
  Rate limited, w

In [ ]:
# # --- Generate extra LLM-labelled training data for the small model (02d) ---
# # Run rank_songs_cot_a on additional generated prompts so 02d has enough data
# if LLM_READY:
#     PROMPT_GEN_TEMPLATE = '''Generate {n} diverse text prompts that a tango DJ might use to \
# describe desired music for their next tanda. Each prompt should describe a mood, energy level, \
# tempo, or style preference. Mix Spanish and English terms naturally.

# Return ONLY a JSON array of strings, no markdown:
# ["prompt 1", "prompt 2", ...]'''

#     print('\nGenerating additional prompts for small model training...')
#     client = genai.Client(api_key=GEMINI_API_KEY)
#     text = llm_chat(client, '', PROMPT_GEN_TEMPLATE.format(n=50), max_tokens=3000)
#     if text.startswith('```'):
#         text = text.split('```')[1]
#         if text.startswith('json'):
#             text = text[4:]
#     try:
#         extra_prompts = json.loads(text)
#     except json.JSONDecodeError:
#         print('WARNING: Failed to parse generated prompts')
#         extra_prompts = []

#     print(f'Labelling {len(extra_prompts)} extra prompts via Method 2A...')
#     extra_results = []
#     for p in extra_prompts:
#         result = rank_songs_cot_a(p, df_merged, FEATURE_PROMPT, client, top_k=5)
#         extra_results.append(result)

#     # Save combined training data (test prompt results + extra results)
#     all_cot_training = list(ALL_RESULTS.get('cot_llm_a', [])) + extra_results
#     path = results_dir / 'cot_training_data.json'
#     with open(path, 'w', encoding='utf-8') as f:
#         json.dump(_jsonify(all_cot_training), f, indent=2)
#     print(f'Saved {len(all_cot_training)} total training examples to {path.name}')
# else:
#     # Save just the test prompt results as training data
#     if ALL_RESULTS.get('cot_llm_a'):
#         path = results_dir / 'cot_training_data.json'
#         with open(path, 'w', encoding='utf-8') as f:
#             json.dump(_jsonify(ALL_RESULTS['cot_llm_a']), f, indent=2)
#         print(f"Saved {len(ALL_RESULTS['cot_llm_a'])} training examples to {path.name}")
#     print('LLM not available \u2014 saved existing results only')